In [1]:
import numpy as np
from numpy.random import uniform
from scipy.optimize import minimize
from scipy.stats import qmc
from scipy.stats import norm
import cyipopt
import warnings
import collections
from smt.applications.ego import Evaluator


from smt.surrogate_models import KRG
from smt.design_space import DesignSpace 
from smt.problems.problem import Problem as SMTProblem

In [2]:
class GaussianProcess:
    def __init__(self, ndim, xlimits=None):
        self.ndim = ndim
        self.xlimits = xlimits
        self.training_x = []
        self.training_y = []
        self.trained = False
    
    # Abstract method for computing the mean of the GP at a given input x
    def mean(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP mean

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Mean of GP at x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method mean")
    
    # Abstract method for computing the covariance of the GP at a given input x
    def covariance(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP covariance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        -------
        ndarray[n, n]
           Covariance of GP at w.r.t. x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method covariance")

    # Abstract method for computing the variance of the GP at a given input x
    def variance(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the GP variance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        ------
        ndarray[n, 1]
           Variance of GP at x
        """
        y = np.ndarray((self.ndim, 1))
        for i in range(x.shape[1]):
            y[i][0] = covariance(np.atleast_2d(x[i,:]))[0][0]
        return y
        #return np.atleast_2d(np.diag(covariance(x))).T

    # Retrieves the bounds of the input space if xlimits is provided.
    def get_bounds(self):
        if self.xlimits is None:
            return None
        else:
            return [(self.xlimits[i][0], self.xlimits[i][1]) for i in range(self.ndim)]

    # Abstract method for training the GP
    def train(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        """
        train the GP model

        Parameters
        ---------
        x : ndarray[n, nx]
        y : ndarray[n, 1]

        """
        NotImplementedError("Child class of GaussianProcess should implement method train")

    # Abstract method for computing the gradient of the mean of the GP at a given input x
    def mean_gradient(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the gradien of GP mean

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Gradient of Mean of GP at x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method mean_gradient")
    
    # Abstract method for computing the gradient of variance of the GP at a given input x
    def variance_gradient(self, x: np.ndarray) -> np.ndarray:
        """
        evaluation of the gradient of GP variance

        Parameters
        ---------
        x: ndarray[n, nx]

        Returns
        -------
        ndarray[n, n]
           Gradient of variance of GP at w.r.t. x
        """
        raise NotImplementedError("Child class of GaussianProcess should implement method variance_gradient")


class smtKRG(GaussianProcess):
    def __init__(self, theta, xlimits, ndim, corr="pow_exp", noise0=None, random_state=None):
        super().__init__(ndim, xlimits)
        if random_state is None:
            random_state = 42
        design_space = DesignSpace(xlimits, random_state=random_state)
        if noise0 is None:
            self.surrogatesmt = KRG(design_space=design_space,
                             print_global=False,
                             eval_noise=False,
                             corr=corr)
        else:
            self.surrogatesmt = KRG(design_space=design_space,
                             print_global=False,
                             noise0=noise0,
                             eval_noise=False,
                             corr=corr)
        self.trained = False

    def mean(self, x):
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict mean or variances")
        return self.surrogatesmt.predict_values(x)

    def variance(self, x):
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict mean or variances")
        return self.surrogatesmt.predict_variances(x)

    def train(self, x, y):
        self.training_x = x
        self.training_y = y
        self.surrogatesmt.set_training_values(x, y)
        self.surrogatesmt.train()
        self.trained = True

    def mean_gradient(self, x: np.ndarray) -> np.ndarray:
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict gradient")
        assert (np.size(x,-1) == self.ndim)
        gradient = [
            self.surrogatesmt._predict_derivatives(x, kx) for kx in range(self.ndim) 
        ]
        return np.atleast_2d(gradient).T

    def variance_gradient(self, x: np.ndarray) -> np.ndarray:
        if not self.trained:
            raise ValueError("must train kriging model before utilizing it to predict gradient")
        return self.surrogatesmt.predict_variance_gradient(x)


In [3]:
# A base class for acquisition functions
class acquisition(object):
    def __init__(self, gpsurrogate):
        assert isinstance(gpsurrogate, GaussianProcess) # add something here
        self.gpsurrogate = gpsurrogate
        self.has_gradient = False
    
    # Abstract method to evaluate the acquisition function at x.
    def evaluate(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("Child class of acquisition should implement method evaluate")

    # Abstract method to evaluate the gradient of acquisition function at x.
    def eval_g(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError("Child class of acquisition should implement method evaluate")

# A subclass of acquisition, implementing the Lower Confidence Bound (LCB) acquisition function.
class LCBacquisition(acquisition):
    def __init__(self, gpsurrogate, beta=3.0):
        super().__init__(gpsurrogate)
        self.beta = beta
        self.has_gradient = True

    # Method to evaluate the acquisition function at x.
    def evaluate(self, x : np.ndarray) -> np.ndarray:
        mu = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        return mu - self.beta * np.sqrt(sig2)

    def eval_g(self, x: np.ndarray) -> np.ndarray:
        mu = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        dsig2_dx = self.gpsurrogate.variance_gradient(x)
        dmu_dx = self.gpsurrogate.mean_gradient(x)
        return dmu_dx - 0.5 * self.beta * dsig2_dx / np.sqrt(sig2)

# A subclass of acquisition, implementing the Expected improvement (EI) acquisition function.
class EIacquisition(acquisition):
    def __init__(self, gpsurrogate):
        super().__init__(gpsurrogate)
        self.has_gradient = True

    # Method to evaluate the acquisition function at x.
    def evaluate(self, x : np.ndarray) -> np.ndarray:        
        y_data = self.gpsurrogate.training_y
        y_min = y_data[np.argmin(y_data[:, 0])]

        pred = self.gpsurrogate.mean(x)
        sig = np.sqrt(self.gpsurrogate.variance(x))

        retval = []
        if sig.size == 1 and np.abs(sig) > 1e-12:
            z = (y_min - pred) / sig
            retval = (y_min - pred) * norm.cdf(z) + sig * norm.pdf(z)
            retval *= -1.
        elif sig.size == 1 and np.abs(sig) <= 1e-12:
            retval = 0.0
        elif sig.size > 1:
            raise NotImplementedError("TODO --- Not implemented yet!")

        return retval

    def eval_g(self, x: np.ndarray) -> np.ndarray:
        y_data = self.gpsurrogate.training_y
        y_min = y_data[np.argmin(y_data[:, 0])]

        mean = self.gpsurrogate.mean(x)
        sig2 = self.gpsurrogate.variance(x)
        sig = np.sqrt(sig2)

        grad_EI = None
        if sig.size == 1 and np.abs(sig) > 1e-12:
            dmean_dx = self.gpsurrogate.mean_gradient(x)
            dsig2_dx = self.gpsurrogate.variance_gradient(x)
            dsig_dx = 0.5 * dsig2_dx / sig

            z = (y_min - mean) / sig
            ncdf = norm.cdf(z)
            npdf = norm.pdf(z)
            EI = (y_min - mean) * ncdf + sig * npdf

            dz_dx = -dmean_dx / sig - (y_min - mean) * dsig_dx / sig**2         
            grad_EI = -dmean_dx * ncdf + dsig_dx * npdf
            grad_EI *= -1.
        elif sig.size == 1 and np.abs(sig) <= 1e-12:
            grad_EI = 0.0
        elif sig.size > 1:
            raise NotImplementedError("TODO --- Not implemented yet!")

        return grad_EI

In [4]:
class Problem:
    def __init__(self, ndim, xlimits, name=" ", constraints=[]):
        self.ndim = ndim
        self.xlimits = xlimits
        assert self.xlimits.shape[0] == ndim            
        assert isinstance(name, str)
        assert isinstance(constraints, collections.abc.Sequence)
        self.name = name
        self.sampler = qmc.LatinHypercube(ndim)
        self.constraints = constraints
            
    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        """
        problem evaluation y = f(x) of
        a scalar valued function f

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Function values
        """
        raise NotImplementedError("Child class of hiopProblem should implement method _evaluate")

    def evaluate(self, x: np.ndarray) -> np.ndarray:
        """
        problem callback y = f(x) of
        the scalar valued function  f

        Parameters
        ---------
        x : ndarray[n, nx]

        Returns
        -------
        ndarray[n, 1]
           Function values (cast to reals)
        """
        y = np.ndarray((x.shape[0], 1))
        y[:,:] = self._evaluate(x)
        return y

    def sample(self, nsample: int) -> np.ndarray:
        """
        generate nsample samples from domain defined
        by xlimits

        Parameters
        -------
        nsample : int

        Returns
        -------
        ndarray[nsample, nx]
           Samples from domain defined by xlimits
        """

        # uniform
        # xsample = np.zeros((nsample, self.ndim))
        # for j in range(self.ndim):
        #    xsample[:, j] = uniform(self.xlimits[j][0], self.xlimits[j][1], size=nsample)

        # from predefined sampler
        xsample = self.sampler.random(nsample)
        xsample = self.xlimits[:,0] + (self.xlimits[:,1] - self.xlimits[:,0]) * xsample

        return xsample

    def set_constraints(self, constraints):
        self.constraints = constraints

class LpNormProblem(Problem):
    def __init__(self, ndim, xlimits, p=2.0, constraints=[]):
        name = "LpNormProblem"
        super().__init__(ndim, xlimits, name=name, constraints=constraints)
        self.p = p

    def _evaluate(self, x):
        ne, nx = x.shape
        assert nx == self.ndim
        y = np.zeros((ne, 1))
        ytemp = np.linalg.norm(x, ord=self.p, axis=1)
        if len(ytemp.shape) == 1:
            y[:,0] = ytemp[:]
        elif len(ytemp.shape) == 2:
            y[:,:] = ytemp[:,:]
        return y

class BraninProblem(Problem):
    def __init__(self, constraints=[]):
        ndim = 2
        xlimits = np.array([[-5.0, 10], [0.0, 15]]) 
        name = 'Branin'
        super().__init__(ndim, xlimits, name=name, constraints=constraints)
            
    def _evaluate(self, x: np.ndarray) -> np.ndarray:
        
        ne, nx = x.shape
        assert nx == self.ndim
        
        y = np.zeros((ne, 1), complex)
        b = 5.1 / (4.0 * (np.pi) ** 2)
        c = 5.0 / np.pi
        r = 6.0
        s = 10.0
        t = 1.0 / (8.0 * np.pi)
        
        arg1 = (x[:,1] - b * x[:,0]**2 + c * x[:,0] - r)
        y[:,0] = arg1**2 + s * (1 - t) * np.cos(x[:,0]) + s
        
        return y


In [5]:
from typing import Callable, Dict, List, Union, Tuple
from scipy.optimize import NonlinearConstraint

class IpoptProb:
    def __init__(self, objective, gradient, constraint:Union[Dict, List[Dict]], xbounds, solver_options=None):
        self.cons = constraint
        self.eval_f = objective
        self.eval_g  = gradient
        self.xl = [b[0] for b in xbounds]
        self.xu = [b[1] for b in xbounds]
        self.cl = []
        self.cu = []
        self.nvar = len(xbounds)
        
        self.ipopt_options = solver_options

        if isinstance(self.cons, list):
            # constraints is provided as a list of dict, supported by SLSQP and Ipopt
            for con in self.cons:
                check_required_keys(con,['type', 'fun'])
                if con['type'] == 'eq':
                    self.cl.append(0.0)
                    self.cu.append(0.0)
                elif con['type'] == 'ineq':
                    self.cl.append(0.0)
                    self.cu.append(np.inf)
                else:
                    raise ValueError(f"Unknown constraint type: {con['type']}")
        elif isinstance(self.cons, dict):
            check_required_keys(self.cons,['cons', 'jac', 'cl', 'cu'])
            # constraints is provided as a dict, supported by trust-constr and Ipopt
            self.cl = constraint['cl']
            self.cu = constraint['cu']
        else:
            raise ValueError("constraints must be provided as a dict of a list of dict.")
        self.ncon = len(self.cl)
        self.nlp = cyipopt.Problem(
                            n=self.nvar,
                            m=self.ncon,
                            problem_obj=self,
                            lb=self.xl,
                            ub=self.xu,
                            cl=self.cl,
                            cu=self.cu
                        )

    def objective(self, x):
        return self.eval_f(x)

    def gradient(self, x):
        return self.eval_g(x)

    def constraints(self, x):
        if isinstance(self.cons, list):
            return np.array([con['fun'](x) for con in self.cons])
        else:
            return self.cons['cons'](x)

    def jacobian(self, x):
        if isinstance(self.cons, list):
            jacs = []
            for con in self.cons:
                if 'jac' in con:
                    jacs.append(con['jac'](x))
                else:
                    raise ValueError("Jacobian not provided for constraint.")
            return np.vstack(jacs)
        else:
            return self.cons['jac'](x)

    def solve(self, x0, solver_options=None):
        ipopt_options = self.ipopt_options
        if solver_options is not None:
            ipopt_options = solver_options
        if ipopt_options is not None:
            for key, value in ipopt_options.items():
                self.nlp.add_option(key, value)

        # Solve the optimization problem
        return self.nlp.solve(x0)


#### BO

In [6]:
# A base class defining a general framework for Bayesian Optimization
class BOAlgorithmBase:
    def __init__(self):
        self.acquisition_type = "LCB" # Type of acquisition function (default = "LCB")
        self.batch_type = "KB"        # strategy for qEI
        self.xtrain = None            # Training data
        self.ytrain = None            # Training data
        self.prob   = None            # Problem structure
        self.evaluator = Evaluator()  # compute control objective evaluations
        self.bo_maxiter = 20          # Maximum number of Bayesian optimization steps
        self.n_start = 10             # estimating acquisition global optima by determining local optima n_start times and then determining the discrete max of that set
        self.batch_size = 1           # batch size
        # save some internal member train
        self.y_hist = None            # History of evaluations
        self.x_hist = None            # History of evaluations
        self.x_opt = None             # Best observed point
        self.y_opt = None             # Best observed value
        self.idx_opt = None           # Index of the best observed value in the history

    # Sets the acquisition function type and batch size
    def setAcquisitionType(self, acquisition_type, batch_size=1):
        self.acquisition_type = acquisition_type
        self.batch_size = batch_size

    # Sets the training data
    def setTrainingData(self, xtrain, ytrain):
        self.xtrain = xtrain
        self.ytrain = ytrain

    # Method to perform Bayesian optimization
    def optimize(self, fun):
        assert NotImplementedError("Child class of hiopEGO should implement method optimize")

    # Method to return the recorded optimization iterations and objectives
    def getOptimizationHistory(self):
        x_hist = np.array(self.x_hist, copy=True)
        y_hist = np.array(self.y_hist, copy=True)
        return x_hist, y_hist

    # Method to return the optimal solution 
    def getOptimalPoint(self):
        x_opt = np.array(self.x_opt, copy=True)
        return x_opt

    # Method to return the optimal objective
    def getOptimalObjective(self):
        y_opt = np.array(self.y_opt, copy=True)
        return y_opt

# A subclass of BOAlgorithmBase implementing a full Bayesian Optimization workflow
class BOAlgorithm(BOAlgorithmBase):
    def __init__(self, prob:Problem, gpsurrogate:GaussianProcess, xtrain, ytrain,
                 user_grad = None,
                 options = {}):
        super().__init__()
        
        assert isinstance(gpsurrogate, GaussianProcess)
        
        self.setTrainingData(xtrain, ytrain)
        self.prob = prob
        self.gpsurrogate = gpsurrogate
        self.bounds = self.gpsurrogate.get_bounds()
        self.fun_grad = None

        self.bo_maxiter = options.get('bo_maxiter', self.bo_maxiter)
        assert self.bo_maxiter > 0, f"Invalid bo_maxiter: {self.bo_maxiter }"
        
        self.solver_options = {"maxiter": 200}
        self.solver_options = options.get('solver_options', self.solver_options)

        acquisition_type = options.get('acquisition_type', "LCB")
        assert acquisition_type in ["LCB", "EI"], f"Invalid acquisition_type: {acquisition_type}"

        batch_size = options.get('batch_size', 1)
        assert isinstance(batch_size, int), f"batch_size {batch_size} not an integer"
        assert batch_size > 0, f"batch_size {batch_size} is not strictly positive"
        self.setAcquisitionType(acquisition_type, batch_size)

        self.evaluator = options.get('evaluator', self.evaluator)
        assert isinstance(self.evaluator, Evaluator)

        acqf_method = options.get('acquisition_method', "multi_start") 
        assert acqf_method in ["multi_start", "bnb"], f"Invalid acqf_method: {acqf_method}"
        self.acqf_method = acqf_method

        if options and 'opt_solver' in options:
            opt_solver = options['opt_solver']
            assert opt_solver in ["SLSQP", "trust-constr", "IPOPT"], f"Invalid opt_solver: {opt_solver}"
        else:
            opt_solver = "SLSQP"

        if isinstance(prob.constraints, dict):
            assert opt_solver in ["trust-constr", "IPOPT"], f"Invalid opt_solver: {opt_solver} while constraints are defined as a dict"
        elif isinstance(prob.constraints, list):
            assert opt_solver in ["SLSQP", "IPOPT"], f"Invalid opt_solver: {opt_solver} while constraints are defined as a list of dict"

        self.set_method(opt_solver)

        if user_grad:
            self.fun_grad = user_grad


    # Method to set up a callback function to minimize the acquisition function
    def _setup_acqf_minimizer_callback(self):
        self.acqf_minimizer_callback = lambda fun, x0: minimizer(fun, x0, self.opt_solver, self.bounds, self.prob.constraints, self.solver_options)

    # Method to train the GP model
    def _train_surrogate(self, x_train, y_train):
        self.gpsurrogate.train(x_train, y_train)

    # Method to find the best next sampling point via optimizing the acquisition function
    def _find_best_point(self, x_train, y_train, x0 = None):

        self._train_surrogate(x_train, y_train)

        if self.acquisition_type == "LCB": #
            acqf = LCBacquisition(self.gpsurrogate)
            self.beta = acqf.beta   

        elif self.acquisition_type == "EI":
            acqf = EIacquisition(self.gpsurrogate)
        else:
            raise NotImplementedError("No implemented acquisition_type associated to" + +self.acquisition_type)

        acqf_obj_callback = lambda x: float(np.array(acqf.evaluate(np.atleast_2d(x))).flat[0])
        acqf_callback = {'obj': acqf_obj_callback}
        
        if acqf.has_gradient == True:
            acqf_grad_callback = lambda x: np.array(acqf.eval_g(np.atleast_2d(x)))
            acqf_callback['grad'] = acqf_grad_callback

        if self.acqf_method == "multi_start":
        
            x_all = []
            y_all = []

            for ii in range(self.n_start):
                success = False
                # Generate random starting point if x0 is not provided
                if self.prob is not None:
                    x0 = self.prob.sample(1)[0]
                else:
                    x0 = np.array([uniform(b[0], b[1]) for b in self.bounds])

                xopt, yout, success = self.acqf_minimizer_callback(acqf_callback, x0)

                if success:
                    x_all.append(xopt)
                    y_all.append(yout)

            if not x_all:

                raise RuntimeError("Optimization failed for all initial points — no solution found.")

            best_xopt = x_all[np.argmin(np.array(y_all))]

            return best_xopt
        
        elif self.acqf_method == "bnb":

            l_init = np.array([b[0] for b in self.bounds])
            u_init = np.array([b[1] for b in self.bounds])

            # Instantiate BnB with GP surrogate and BO callback
            bnb = BnBAlgorithm(
                gpsurrogate=self.gpsurrogate,
                acq_minimizer_callback=self.acqf_minimizer_callback
            )
            bnb.acquisition_type = self.acquisition_type
            bnb.beta = self.beta

            # Run BnB optimization
            best_l, best_u, _ = bnb.optimize(x_train, y_train, l_init, u_init)

            # Take the midpoint of the best box as the candidate point
            x_best = 0.5 * (best_l + best_u)
            return x_best
            
    def _get_virtual_point(self, x):

        if self.batch_type not in ["CLmin", "KB", "KBUB", "KBLB", "KBRand"]:
            raise NotImplementedError("No implemented batch_type associated to"+self.batch_type)
        
        # constant-liar, Kriging-believer and Kriging-believer variants
        if self.batch_type == "CLmin":
            return min(self.gpsurrogate.training_y)
        elif self.batch_type == "KB":
            beta = 0.
        elif self.batch_type == "KBUB":
            beta = 3.0
        elif self.batch_type == "KBLB":
            beta = -3.0
        elif self.batch_type == "KBRand":
            beta = np.random.randn()
        return self.gpsurrogate.mean(x) + beta * np.sqrt(self.gpsurrogate.variance(x))

    # Set the optimization method
    def set_method(self, method):
        self.opt_solver = method

    # Set the options for the internal optimization solver
    def set_options(self, solver_options):
        self.solver_options = solver_options

    # Method to perform Bayesian optimization
    def optimize(self):
      x_train = self.xtrain
      y_train = self.ytrain
      
      n_init_sample = np.size(x_train, 0)
      self._setup_acqf_minimizer_callback()

      self.x_hist = []
      self.y_hist = []

      for i in range(self.bo_maxiter):
          print(f"*****************************")
          print(f"Iteration {i+1}/{self.bo_maxiter}")

          y_train_virtual = y_train.copy() # old training + batch_size num of virtual points
          for j in range(self.batch_size):
             # Get a new sample point
             x_new = self._find_best_point(x_train, y_train_virtual)
             
             # Update training sample points
             x_train         = np.vstack([x_train,         x_new    ])

             # if this is not the last point in the current batch
             # then obtain a virtual point
             if j < max(range(self.batch_size)):
                 # Get a virtual point
                 y_virtual = self._get_virtual_point(np.atleast_2d(x_new))

                 # Update training set with the virtual point
                 y_train_virtual = np.vstack([y_train_virtual, y_virtual])
          
          y_new = self.evaluator.run(self.prob.evaluate, x_train[-self.batch_size:])
          y_train = np.vstack([y_train, y_new])
          #print(f'x_new = {x_new}')
          #print(f'y_new = {y_new}')
          
          # Save the new sample points and objective evaluations
          for j in range(1, self.batch_size+1):
              self.x_hist.append(x_train[-j].flatten())
              self.y_hist.append(y_train[-j].flatten())
          if self.batch_size == 1:
              print(f"Sample point X: {x_train[-self.batch_size:]}, Observation Y: {y_new}")
          else:
              print(f"Sample points X: {x_train[-self.batch_size:]}, Observations Y: {y_new}")


      # Save the optimal results and all the training data
      self.idx_opt = np.argmin(self.y_hist)
      self.x_opt = self.x_hist[self.idx_opt]
      self.y_opt = self.y_hist[self.idx_opt]
      self.setTrainingData(x_train, y_train)

      print(f"\n\nOptimal at BO iteration: {self.idx_opt+1} ")
      #if self.idx_opt < n_init_sample:
      #    print(f"Optimal at initial sample: {self.idx_opt+1}")
      #else:
      #    print(f"Optimal at BO iteration: {self.idx_opt-n_init_sample+1} ")
          
      print(f"Optimal point: {self.x_opt.flatten()}, Optimal value: {self.y_opt}")
      print()
    
def minimizer(fun, x0, method, bounds, constraints, solver_options):
    if method == "SLSQP":
        if 'grad' in fun:
            y = minimize(fun['obj'], x0, method=method, bounds=bounds, jac=fun['grad'], constraints=constraints, options=solver_options)
        else:
            y = minimize(fun['obj'], x0, method=method, bounds=bounds, constraints=constraints, options=solver_options)
        success = y.success
        if not success:
            print(y.message)
        xopt = y.x
        yopt = y.fun
    elif method == "trust-constr":
        nonlinear_constraint = NonlinearConstraint(constraints['cons'], constraints['cl'], constraints['cu'], jac=constraints['jac'])
        y = minimize(fun['obj'], x0, method=method, bounds=bounds, constraints=[nonlinear_constraint], options=solver_options)
        success = y.success
        if not success:
            print(y.message)
        xopt = y.x
        yopt = y.fun
    else:
        ipopt_prob = IpoptProb(fun['obj'], fun['grad'], constraints, bounds, solver_options)
        print(x0)
        sol, info = ipopt_prob.solve(x0)

        status = info.get('status', -999)
        msg = info.get('status_msg', b'unknown error')
        if status == 0:
            # ipopt returns 0 as success
            success = True
        else:
            warnings.warn(f"Ipopt failed to solve the problem. Status msg: {msg}")
            success = False

        yopt = info['obj_val']
        xopt = sol

    return xopt, yopt, success

#### BnB

In [7]:
import numpy as np
import cvxpy as cp

class BnBAlgorithmBase:
    def __init__(self, x=None, y=None):
        # Node class for priority queue
        self.BnBNode = BnBNode
        self.BnB_LBmethod = None

        # Stopping criteria
        self.epsilon_gap = 1e-3
        self.epsilon_diam = 1e-2

        # Kernel info for bounds
        self.kernel_spec = None
        self.kernel_func = None
        self.y_min = None

        # Evaluation parameters
        self.K = None
        self.K_inv = None
        self.theta = 1.0

        # BnB search state
        self.best_l = None
        self.best_u = None
        self.upper_bound = np.inf
        self.final_gap = None
        self.final_diameter = None
        self.total_nodes = 0
        self.verbose = False

        # Training data
        self.x = x
        self.y = y

    def sync_kernel_from_surrogate(self):
        
        corr_map = {
            "pow_exp": "pow_exp",
            "abs_exp": "abs_exp",
            "matern32": "matern32",
            "matern52": "matern52"
        }
        corr_type = self.gpsurrogate.surrogatesmt.options['corr']
        self.theta = self.gpsurrogate.surrogatesmt.corr.theta
        self.set_kernel(corr_map[corr_type])

    def set_kernel(self, kernel_spec):

        assert kernel_spec in ["abs_exp", "pow_exp", "matern32", "matern52"]
        self.kernel_spec = kernel_spec

        if kernel_spec == "abs_exp":  # ν = 1/2
            self.kernel_func = lambda d: np.exp(-np.sqrt(d))

        elif kernel_spec == "pow_exp":  # SE, ν = ∞
            self.kernel_func = lambda d: np.exp(-d / 2)

        elif kernel_spec == "matern32":  # ν = 3/2
            self.kernel_func = lambda d: (
                (1 + np.sqrt(3) * np.sqrt(d)) *
                np.exp(-np.sqrt(3) * np.sqrt(d))
            )

        elif kernel_spec == "matern52":  # ν = 5/2
            self.kernel_func = lambda d: (
                (1 + np.sqrt(5) * np.sqrt(d) + (5/3) * d) *
                np.exp(-np.sqrt(5) * np.sqrt(d))
            )



    def set_covmatrix(self, x):
        n = x.shape[0]
        self.K = np.zeros((n, n))
        
        # Compute symmetric kernel matrix
        for i in range(n):
            for j in range(i, n):
                d = np.sum(((x[i] - x[j]) / self.theta)**2) 
                self.K[i, j] = self.kernel_func(d)
                self.K[j, i] = self.K[i, j]
        
        # --- Debug: print raw K ---
        print("\n=== Covariance Matrix K ===")
        print(self.K)
        
        # --- Add jitter for numerical stability ---
        jitter = 1e-8
        self.K += jitter * np.eye(n)
        
        # --- Debug: eigenvalues & condition number ---
        eigvals = np.linalg.eigvalsh(self.K)
        cond_number = np.linalg.cond(self.K)
        print("Eigenvalues of K:", eigvals)
        print("Condition number of K:", cond_number)
        
        # --- Compute inverse ---
        try:
            self.K_inv = np.linalg.inv(self.K)
            print("K_inv:\n", self.K_inv)
        except np.linalg.LinAlgError:
            print("ERROR: Singular K detected, cannot invert.")
            self.K_inv = np.linalg.pinv(self.K)  # fallback to pseudo-inverse
        
        return self.K_inv

    def ker_bounds(self, x, l, u):

    
        kL, kU = [], []

        for i in range(len(x)):

            # Lower distance (closest point to box)
            d_L = np.sum((np.maximum(0, np.maximum(l - x[i], x[i] - u)) / self.theta)**2)
            # Upper distance (farthest point to box)
            d_U = np.sum((np.maximum(np.abs(l - x[i]), np.abs(u - x[i])) / self.theta)**2)

            # Lower kernel at largest distance
            kL.append(self.kernel_func(d_U))  
            # Upper kernel at smallest distance
            kU.append(self.kernel_func(d_L))  

        return np.array(kL), np.array(kU)


    def mu_bounds(self, y, kL, kU):

        alpha = self.K_inv @ y
        mu_U = np.sum(alpha * np.where(alpha >= 0, kU, kL))
        mu_L = np.sum(alpha * np.where(alpha >= 0, kL, kU))

        return mu_L, mu_U

    def sigma2_U(self, kL, kU):

        # Set up QP to solve for upper variance bound
        var = cp.Variable(len(kU))
        obj = cp.Maximize(1 - cp.quad_form(var, self.K_inv))
        constraints = [var >= kL, var <= kU]
        prob = cp.Problem(obj, constraints)
        prob.solve(solver=cp.OSQP)

        sigma2_U = prob.value
        return max(sigma2_U, 0)

    def sigma2_L(self,kL, kU, epsilon = 1e-6, random_seed=1034):

        # Randomly initialize a point in the bounds
        np.random.seed(random_seed)
        var = np.random.uniform(kL, kU)

        # Initialize active coordinates
        active_coords = set(range(len(kL)))

        # Define the function to minimize
        def f(k_vec): return 1 - k_vec @ self.K_inv @ k_vec
        f_curr = f(var)

        # Iteratively improve the point by evaluating each coordinate direction.
        while active_coords:
            improvement = False
            for i in list(active_coords):
                for val in [kL[i], kU[i]]:
                    var_new = var.copy()
                    var_new[i] = val
                    f_val = f(var_new)
                    if f_val < f_curr - epsilon:
                        var = var_new
                        f_curr = f_val
                        improvement = True
                        break
                if not improvement:
                    active_coords.remove(i)
            if not improvement:
                break

        sigma2_L = f_curr
        return max(sigma2_L, 0)

    def rs_ei(self, y, mu, sigma):
        
        y_min = np.min(y)

        if sigma > 1e-12:
            z = (y_min - mu) / sigma
            ei = (y_min - mu) * norm.cdf(z) + sigma * norm.pdf(z)
            return -ei
        else:
            # Deterministic case: EI = max(y_min - mu, 0)
            return -max(y_min - mu, 0.0)
    
    def rs_lcb(self, mu, sigma):

        return mu - self.beta * sigma


In [8]:
import heapq
import collections
from scipy.stats import norm
from scipy.optimize import minimize

class BnBNode:
    def __init__(self, l, u, aq_L, aq_U):
        self.l = l
        self.u = u
        self.aq_L = aq_L
        self.aq_U = aq_U
        self.diam = np.max(u - l)
        self.midpoint = 0.5 * (l + u)

    def __lt__(self, other):
        return self.aq_U > other.aq_U

class BnBAlgorithm(BnBAlgorithmBase):
    def __init__(self, gpsurrogate, acq_minimizer_callback=None):
        super().__init__()  # no args
        self.gpsurrogate = gpsurrogate
        self.acq_minimizer_callback = acq_minimizer_callback
        self.sync_kernel_from_surrogate()

    def _branch(self, l, u):

        # Force to float to avoid truncation issues
        l = l.astype(float)
        u = u.astype(float)

        # Pick the dimension with largest length
        d = np.argmax(u - l)
        mid = 0.5 * (l[d] + u[d])

        # If the midpoint is the same as one bound (degenerate split), return nothing
        if np.isclose(mid, l[d]) or np.isclose(mid, u[d]):
            return []

        # Generate child boxes
        l1, u1 = l.copy(), u.copy()
        l2, u2 = l.copy(), u.copy()
        
        # Split along midpoint
        u1[d] = mid
        l2[d] = mid

        return [(l1, u1), (l2, u2)]


    # For minimization, we find a feasible function value as the upper bound on the minimum value of the acquisition function.
    def compute_acq_upper_bound(self, x, y, l, u):

            if self.BnB_LBmethod == "IPOPT":
                    
                    if self.acquisition_type == "LCB":

                        acqf = LCBacquisition(self.gpsurrogate)

                    elif self.acquisition_type == "EI":

                        acqf = EIacquisition(self.gpsurrogate)

                    else:
                        raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

                    acqf_obj_callback = lambda x: float(np.array(acqf.evaluate(np.atleast_2d(x))).flat[0])
                    acqf_callback = {'obj': acqf_obj_callback}
                    if acqf.has_gradient == True:
                                acqf_grad_callback = lambda x: np.array(acqf.eval_g(np.atleast_2d(x)))
                                acqf_callback['grad'] = acqf_grad_callback

                    if x0 is None:
                        x0 = np.array(uniform(l, u))
                    
                    xopt, yout, success = self.acqf_minimizer_callback(acqf_callback, x0)

                    if not success:
                        raise RuntimeError("EI maximization failed")

                    return xopt, yout
            else:
                
                # We compute the upper bound of the acquisition function based on bounds of the kernel, mu and sigma.
                
                # Compute the kernel bounds with given x
                kL, kU = self.ker_bounds(x, l, u)
                # Compute the mean bounds
                mu_L, mu_U = self.mu_bounds(y,kL, kU)
                var_L = self.sigma2_L(kL, kU)

                if self.acquisition_type == "LCB":

                    lcb_U = self.rs_lcb(mu_U, np.sqrt(var_L))
                    return lcb_U
                
                elif self.acquisition_type == "EI":
                     
                    ei_U = self.rs_ei(y, mu_U, np.sqrt(var_L))
                    return ei_U
                
                else:
                    raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

    # For minimization, we compute the lower bound explicitly using the acquisition function over mu, sigma.
    def compute_acq_lower_bound(self, x,y,l,u):

                # We compute the upper bound of the acquisition function based on bounds of the kernel, mu and sigma.
                
                # Compute the kernel bounds with given x
                kL, kU = self.ker_bounds(x, l, u)
                # Compute the mean bounds
                mu_L, mu_U = self.mu_bounds(y, kL, kU)
                var_U = self.sigma2_U(kL, kU)

                if self.acquisition_type == "LCB":

                    lcb_U = self.rs_lcb(mu_L, np.sqrt(var_U))
                    return lcb_U
                
                elif self.acquisition_type == "EI":
                     
                    ei_U = self.rs_ei(y, mu_L, np.sqrt(var_U))
                    return ei_U
                
                else:
                    raise NotImplementedError("No implemented acquisition_type associated to" + self.acquisition_type)

    def optimize(self, x, y, l_init, u_init):
        """
        Branch & Bound minimization with tolerance stopping.
        """

        print("=== Starting Branch & Bound Optimization (Minimization) ===")
        print(f"Initial bounds: l = {l_init}, u = {u_init}")
        print(f"Number of points: {x.shape[0]}, Dim = {x.shape[1]}")

        # --- Compute kernel and inverse ---
        self.K_inv = self.set_covmatrix(x)
        eigvals = np.linalg.eigvalsh(self.K)
        print("\nCovariance Matrix K:")
        print(self.K)
        print("Eigenvalues of K:", eigvals)
        print("Rank of K:", np.linalg.matrix_rank(self.K))

        # --- Compute initial acquisition bounds ---
        aq_L_val = self.compute_acq_lower_bound(x, y, l_init, u_init)
        aq_U_val = self.compute_acq_upper_bound(x, y, l_init, u_init)
        print(f"\nInitial acquisition lower bound: {aq_L_val}")
        print(f"Initial acquisition upper bound: {aq_U_val}")

        # --- Initialize root node ---
        root = BnBNode(l_init.astype(float), u_init.astype(float), aq_L_val, aq_U_val)
        queue = [root]
        heapq.heapify(queue)  # should be ordered by lower bound for minimization

        # Global best feasible value (upper bound for minimization)
        self.best_val = aq_U_val
        self.best_l, self.best_u = l_init.copy(), u_init.copy()

        # Track diameters and iteration count
        diameters = [np.max(u_init - l_init)]
        self.total_nodes = 1
        iteration = 0

        # --- Branch & Bound Loop ---
        while queue:
            iteration += 1
            node = heapq.heappop(queue)  # Pop node with smallest lower bound

            # --- Update best feasible solution ---
            if node.aq_U < self.best_val:
                self.best_val = node.aq_U
                self.best_l, self.best_u = node.l, node.u

            print(f"\n--- Iteration {iteration} ---")
            print(f"Node bounds: l={node.l}, u={node.u}")
            print(f"Node acquisition bounds: L={node.aq_L}, U={node.aq_U}")
            print(f"Current best feasible value (GUB): {self.best_val}")

            # --- Per-node optimality check (mirrors maximization code) ---
            # Stop if best feasible (GUB) is within epsilon of this node's lower bound
            if self.best_val - node.aq_L <= self.epsilon_gap:
                print(f"STOPPING: GUB - Node.L = {self.best_val - node.aq_L} <= {self.epsilon_gap}")
                break

            # --- Diameter-based early stopping ---
            node_diameter = np.max(node.u - node.l)
            if node_diameter <= self.epsilon_diam:
                print(f"STOPPING: Node diameter {node_diameter} <= {self.epsilon_diam}")
                break

            # --- Node-level pruning with tolerance ---
            if node.aq_L >= self.best_val - self.epsilon_gap:
                print("Pruned: Node cannot improve the best value within tolerance.")
                continue

            # --- Branch into children ---
            for l_child, u_child in self._branch(node.l, node.u):
                print(f"  Branching to child: l={l_child}, u={u_child}")

                aq_L_r = self.compute_acq_lower_bound(x, y, l_child, u_child)
                aq_U_r = self.compute_acq_upper_bound(x, y, l_child, u_child)
                print(f"  Child acquisition bounds: L={aq_L_r}, U={aq_U_r}")

                self.total_nodes += 1
                diameters.append(np.max(u_child - l_child))

                # Prune children that cannot improve the current best feasible
                if aq_L_r >= self.best_val:
                    print("  Child pruned: Lower bound ≥ current best feasible value.")
                    continue

                # Update best feasible if we found a better upper bound
                if aq_U_r < self.best_val:
                    self.best_val = aq_U_r
                    self.best_l, self.best_u = l_child, u_child

                # Push promising child to queue
                heapq.heappush(queue, BnBNode(l_child, u_child, aq_L_r, aq_U_r))

            # --- Queue empty check ---
            if not queue:
                print("\nSTOPPING: Queue empty, no better nodes remain.")
                break

            # --- Global gap check (mirrors maximization formula) ---
            phi_GUB = self.best_val                       # Best feasible
            phi_GLB = min(n.aq_L for n in queue)          # Min lower bound in queue
            global_gap = phi_GUB - phi_GLB

            if global_gap <= self.epsilon_gap:
                print(f"\nSTOPPING: GUB - GLB = {global_gap} <= {self.epsilon_gap}")
                break

        # --- Final stats ---
        self.final_gap = self.best_val - min([n.aq_L for n in queue], default=self.best_val)
        self.final_diameter = min(diameters)

        print("\n=== Optimization Finished ===")
        print(f"Total nodes explored: {self.total_nodes}")
        print(f"Best bounds: l={self.best_l}, u={self.best_u}")
        print(f"Best feasible acquisition value (GUB): {self.best_val}")
        print(f"Final gap: {self.final_gap}, final diameter: {self.final_diameter}")

        return self.best_l, self.best_u, self.best_val


In [9]:
def check_required_keys(user_dict, required_keys):
    for key in required_keys:
        if key not in user_dict:
            raise KeyError(f"Missing required key: '{key}'")

In [10]:
# Get user input for the number of repetitions from command-line arguments
num_repeat = 1

### parameters
n_samples = 5  # number of the initial samples to train GP
theta = 1.e-2  # hyperparameter for GP kernel
nx = 2         # dimension of the problem
xlimits = np.array([[-5, 5], [-5, 5]]) # bounds on optimization variable

prob_type_l = ["LpNorm", "Branin"]
acq_type_l = ["LCB", "EI"]

prob_type_l = ["LpNorm"]
acq_type_l = ["LCB"]

def con_eq(x):
  return  x[0]**2 + x[1]**2 - 1

def con_jac_eq(x):
  return  np.array([2*x[0], 2*x[1]])

def con_ineq(x):
  return  x[0] - x[1]

def con_jac_ineq(x):
  return  np.array([1.0, -1.0])

# 'SLSQP' requires constraints defined in a list of dict
user_constraint_list = [{'type': 'eq','fun': con_eq,   'jac': con_jac_eq},
                   {'type': 'ineq', 'fun': con_ineq, 'jac': con_jac_ineq}]

def cons_vec(x):
    x1, x2 = x
    return np.array([
        (x1 - 2)**2 + (x2 - 2.5)**2 - 2,
        x1 + x2 - 5,
        -x1
    ])

# Jacobian of constraints
def cons_jac_vec(x):
    x1, x2 = x
    return np.array([
        [2 * (x1 - 2), 2 * (x2 - 2.5)],
        [1, 1],
        [-1, 0]
    ])

cl = -np.inf * np.ones(3)
cu = np.zeros(3)

# 'trust-constr' method supports vector-valued constraints
user_constraint_dict = {'cons': cons_vec, 'jac': cons_jac_vec, 'cl': cl, 'cu': cu}


In [11]:
retval = 0
for prob_type in prob_type_l:
   print()
   if prob_type == "LpNorm":
      problem = LpNormProblem(nx, xlimits)
   else:
      problem = BraninProblem()
   problem.set_constraints(user_constraint_dict)

   for acq_type in acq_type_l:
      print("Problem name: ", problem.name)
      print("Acquisition type: ", acq_type)

      ### initial training set
      x_train = problem.sample(n_samples)
      y_train = problem.evaluate(x_train)

      ### Define the GP surrogate model
      gp_model = smtKRG(theta, xlimits, nx)
      gp_model.train(x_train, y_train)
      print(f"GP model trained with theta = {theta}")

      options = {
        'acquisition_type': acq_type,
        'acquisition_method': 'bnb',
        'bo_maxiter': 3,
        'opt_solver': 'IPOPT', #"SLSQP" "IPOPT" "trust-constr"
        'solver_options': {
        #   'maxiter': 100 #,'print_level': 5
           }
      }
   
      # Instantiate and run Bayesian Optimization
      bo = BOAlgorithm(problem, gp_model, x_train, y_train, options = options) #EI or LCB
      bo.optimize()

#sys.exit(retval)



Problem name:  LpNormProblem
Acquisition type:  LCB
GP model trained with theta = 0.01
*****************************
Iteration 1/3
=== Starting Branch & Bound Optimization (Minimization) ===
Initial bounds: l = [-5 -5], u = [5 5]
Number of points: 5, Dim = 2

=== Covariance Matrix K ===
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]]
Eigenvalues of K: [1.00000001 1.00000001 1.00000001 1.00000001 1.00000001]
Condition number of K: 1.0
K_inv:
 [[0.99999999 0.         0.         0.         0.        ]
 [0.         0.99999999 0.         0.         0.        ]
 [0.         0.         0.99999999 0.         0.        ]
 [0.         0.         0.         0.99999999 0.        ]
 [0.         0.         0.         0.         0.99999999]]

Covariance Matrix K:
[[1.00000001 0.         0.         0.         0.        ]
 [0.         1.00000001 0.         0.         0.        ]
 [0.         0.         1.00000001 0.         0.        ]
 [0.         0.        

In [12]:
retval = 0
for prob_type in prob_type_l:
   print()
   if prob_type == "LpNorm":
      problem = LpNormProblem(nx, xlimits)
   else:
      problem = BraninProblem()
   problem.set_constraints(user_constraint_dict)

   for acq_type in acq_type_l:
      print("Problem name: ", problem.name)
      print("Acquisition type: ", acq_type)

      ### initial training set
      x_train = problem.sample(n_samples)
      y_train = problem.evaluate(x_train)

      ### Define the GP surrogate model
      gp_model = smtKRG(theta, xlimits, nx)
      gp_model.train(x_train, y_train)

      options = {
        'acquisition_type': acq_type,
        'acquisition_method': 'multi_start', # "multi_start" "bnb"
        'bo_maxiter': 3,
        'opt_solver': 'IPOPT', #"SLSQP" "IPOPT" "trust-constr"
        'solver_options': {
        #   'maxiter': 100 #,'print_level': 5
           }
      }
   
      # Instantiate and run Bayesian Optimization
      bo = BOAlgorithm(problem, gp_model, x_train, y_train, options = options) #EI or LCB
      bo.optimize()

#sys.exit(retval)



Problem name:  LpNormProblem
Acquisition type:  LCB
*****************************
Iteration 1/3
[3.99352069 0.30882292]
[-3.11356623 -0.10504193]
[-0.01712046 -1.71012312]
[-0.31654459  1.73992423]
[-0.91163562  4.55958809]
******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.17, running with linear solver MUMPS 5.6.2.


Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        6
Number of nonzeros in Lagrangian Hessian.............:        0

Total number of variables............................:        2
                     variables with only lower bounds:     

#### Testing Values of Kernels and Bounds with SMT